In [1]:
import numpy as np
z = np.array([-4., -2., 0., 2., 4.])
sig = 1 / (1 + np.exp(-z))
print(sig.round(4))
print((sig * (1 - sig)).round(4))
print(np.tanh(z).round(4))
# [0.018  0.1192 0.5    0.8808 0.982 ]
# [0.0177 0.105  0.25   0.105  0.0177]
# [-0.9993 -0.964   0.      0.964   0.9993]

[0.018  0.1192 0.5    0.8808 0.982 ]
[0.0177 0.105  0.25   0.105  0.0177]
[-0.9993 -0.964   0.      0.964   0.9993]


In [2]:
import numpy as np
z = np.array([-4., -2., 2., 4.])
print(np.maximum(0, z))
print(np.where(z > 0, z, 0.01 * z))
print(np.where(z < 0, 1.0, 0.01))
# [0. 0. 2. 4.]
# [-0.04 -0.02  2.    4.  ]
# [1.   1.   0.01 0.01]

[0. 0. 2. 4.]
[-0.04 -0.02  2.    4.  ]
[1.   1.   0.01 0.01]


In [3]:
import torch
import torch.nn.functional as F
for f in (F.relu, F.leaky_relu):
  w = torch.tensor(1.0, requires_grad=True)
  h = f(w * 2 - 5)
  ((h - 1) ** 2).backward()
  print(round(h.item(), 4), round(w.grad.item(), 4))
#   0.0 0.0
# -0.03 -0.0412

0.0 0.0
-0.03 -0.0412


In [5]:
import numpy as np
x = np.arange(1., 6.)
y = np.array([2., 4., 6., 8., 40.])
for a, b in ((2, 0), (8, -12)):
  p = a * x + b
  print(a, b, np.mean((y - p) ** 2), np.mean(np.abs(y - p)))
# 2  0  180.0  6.0
# 8 -12 72.0   7.2

2 0 180.0 6.0
8 -12 72.0 7.2


In [7]:
import numpy as np
def softmax(z):
  e = np.exp(z - np.max(z))
  return e / np.sum(e)

print(softmax(np.array([2., 1., 0.])).round(4))
print(softmax(np.array([1000., 1001.])).round(4))

# [0.66524096 0.24472847 0.09003057]
# [0.26894142 0.73105858]

[0.6652 0.2447 0.09  ]
[0.2689 0.7311]


In [8]:
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler as StandartScaler

X, y = load_wine(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
sc = StandartScaler().fit(X_train)
A, B = [torch.tensor(sc.transform(v)).float() for v in (X_train, X_test)]
ya, yb = torch.tensor(y_train), torch.tensor(y_test)
# har doim torch.tensor qilib olish kerak ( bo'lmasa neural network tushunmay qoladi)

print(X.shape, tuple(A.shape), tuple(B.shape))

# (178, 13) (124, 13) (54, 13)

(178, 13) (124, 13) (54, 13)


In [10]:
import torch.nn as nn

ACTS = {
    "sigmoid": nn.Sigmoid,
    "tanh": nn.Tanh,
    "relu": nn.ReLU,
    "leaky": nn.LeakyReLU
}

def fit_eval(act, lr=0.05, seed=42, loss_fn=F.cross_entropy):
  torch.manual_seed(seed)
  a = ACTS[act]
  net = nn.Sequential(nn.Linear(13, 16), a(), nn.Linear(16, 16), a(), nn.Linear(16, 16), a(), nn.Linear(16, 3))

  opt = torch.optim.SGD(net.parameters(), lr=lr)
  for _ in range(100):
    loss = loss_fn(net(A), ya)
    opt.zero_grad()
    loss.backward()
    opt.step()
  return round((net(B).argmax(1) == yb).float().mean().item(), 4)

In [11]:
for lr in (0.05, 0.09):
  for act in ACTS:
    print(lr, act, fit_eval(act, lr))

# 0.05 sigmoid 0.3889
# 0.05 tanh 0.9815
# 0.05 relu 0.9444
# 0.05 leaky 0.963
# 0.09 sigmoid 0.3889
# 0.09 tanh 0.9815
# 0.09 relu 0.9815
# 0.09 leaky 0.9815

0.05 sigmoid 0.3889
0.05 tanh 0.9815
0.05 relu 0.9444
0.05 leaky 0.963
0.09 sigmoid 0.3889
0.09 tanh 0.9815
0.09 relu 0.9815
0.09 leaky 0.9815


In [12]:
import numpy as np
for lr in (0.05, 0.5):
  for act in ACTS:
    a = [fit_eval(act, lr, k) for k in range(5)]
    print(lr, act, round(np.mean(a), 3), min(a), max(a))

# 0.05 sigmoid 0.389 0.3889 0.3889
# 0.05 tanh 0.952 0.9259 0.963
# 0.05 relu 0.656 0.4074 0.8148
# 0.05 leaky 0.656 0.3889 0.8333
# 0.5 sigmoid 0.389 0.3889 0.3889
# 0.5 tanh 0.982 0.9815 0.9815
# 0.5 relu 0.982 0.963 1.0
# 0.5 leaky 0.982 0.963 1.0

0.05 sigmoid 0.389 0.3889 0.3889
0.05 tanh 0.952 0.9259 0.963
0.05 relu 0.656 0.4074 0.8148
0.05 leaky 0.656 0.3889 0.8333
0.5 sigmoid 0.389 0.3889 0.3889
0.5 tanh 0.982 0.9815 0.9815
0.5 relu 0.982 0.963 1.0
0.5 leaky 0.982 0.963 1.0
